# 00 · Environment check

What this node can run: RDMA devices, GIDs, libfabric providers, pyverbs, native builds, tests.

In [1]:
MODE = "local"            # "local" or "cluster" (run on each node)
RESULTS_SUBDIR = "env"
NETDEV = None             # 10GbE interface carrying SoftRoCE, e.g. "enp1s0f0"; used only to print the rxe setup command
RUN_TESTS = True

In [2]:
import os, sys, json, socket, subprocess, time
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "middleware").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
HOST = socket.gethostname()
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
RESULTS = (ROOT / "benchmarks" / "results" / RESULTS_SUBDIR) if MODE == "cluster" else Path("/tmp/scuffed-local") / RESULTS_SUBDIR
RESULTS.mkdir(parents=True, exist_ok=True)
FIGDIR = ROOT / "docs" / "figures"
(FIGDIR / MODE).mkdir(parents=True, exist_ok=True)

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 170, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "legend.fontsize": 7.5, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 1.4,
    "lines.markersize": 4, "figure.constrained_layout.use": False})
C = {"hot": "#f97316", "cold": "#38bdf8", "spill": "#a78bfa", "ok": "#34d399", "bad": "#f43f5e",
     "gold": "#facc15", "ink": "#1f2937", "muted": "#94a3b8"}

def fig(name, model=False, caption=None, f=None):
    """Save the current figure as PNG + PDF and show it.
    model=True: arithmetic or repo results, same on every machine -> docs/figures/<name>
    model=False: measured on this machine -> docs/figures/<MODE>/<name>"""
    f = f or plt.gcf()
    d = FIGDIR if model else FIGDIR / MODE
    f.tight_layout()
    for ext in ("png", "pdf"):
        f.savefig(d / f"{name}.{ext}", bbox_inches="tight")
    print(f"figure -> {d.relative_to(ROOT)}/{name}.png" + (f"  |  {caption}" if caption else ""))
    plt.show()

def save(name, obj):
    p = RESULTS / f"{HOST}_{name}.json"
    obj = {"host": HOST, "utc": STAMP, "mode": MODE, **obj}
    p.write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", p)
    return p

def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=ROOT)
    print(r.stdout[-4000:], r.stderr[-2000:])
    if check and r.returncode:
        raise RuntimeError(f"{cmd} -> {r.returncode}")
    return r
print("repo:", ROOT, "| host:", HOST, "| mode:", MODE, "| results ->", RESULTS, "| figures ->", FIGDIR)

repo: /home/claude/scuffedrdma | host: vm | mode: local | results -> /tmp/scuffed-local/env | figures -> /home/claude/scuffedrdma/docs/figures


Run this first on every node. It records what the node actually has (RDMA devices, GIDs, libfabric providers,
pyverbs, compilers), builds the three native libraries, and runs the test suite. On a node with an RDMA
device the `rdma`-marked tests run too (verbs provider, `verbs;ofi_rxm`, native libibverbs loopback over rxe0).

In [3]:
import platform, shutil
info = {"uname": platform.uname()._asdict(), "python": sys.version, "cpus": os.cpu_count()}
for cmd in ["lscpu | head -20", "nproc", "uname -r"]:
    info[cmd] = sh(cmd, check=False).stdout

$ lscpu | head -20
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) Processor @ 2.80GHz
CPU family:                              6
Model:                                   85
Thread(s) per core:                      1
Core(s) per socket:                      2
Socket(s):                               1
Stepping:                                7
BogoMIPS:                                5600.45
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_

## Hardware of this machine

Written to `docs/figures/<MODE>/HARDWARE.md` so every measured figure has the machine it came from next to it.

In [4]:
import re, json as _json
def _cmd(c):
    r = subprocess.run(c, shell=True, text=True, capture_output=True); return (r.stdout or r.stderr).strip()
hw = {"host": HOST, "mode": MODE, "utc": STAMP}
hw["virtualization"] = _cmd("systemd-detect-virt 2>/dev/null || echo unknown")
hw["os"] = _cmd("grep PRETTY_NAME /etc/os-release | cut -d= -f2").strip(chr(34))
hw["kernel"] = _cmd("uname -r")
lscpu = {k.strip(): v.strip() for k, v in (l.split(":", 1) for l in _cmd("lscpu").splitlines() if ":" in l)}
hw["cpu"] = {k: lscpu.get(k) for k in ("Model name", "CPU(s)", "Thread(s) per core", "Core(s) per socket", "Socket(s)",
                                     "CPU max MHz", "Hypervisor vendor", "L1d cache", "L2 cache", "L3 cache", "NUMA node(s)") if lscpu.get(k)}
mem = {l.split()[0].rstrip(":"): int(l.split()[1]) for l in Path("/proc/meminfo").read_text().splitlines()[:3]}
hw["memory_GiB"] = round(mem.get("MemTotal", 0) / 2**20, 1)
hw["block_devices"] = chr(10).join(l for l in _cmd("lsblk -d -o NAME,SIZE,TYPE,ROTA,MODEL 2>/dev/null").splitlines() if re.search(r"\s\d+(\.\d+)?[GT]\s", l))
hw["tmp_mount"] = _cmd("df -h /tmp | tail -1")
nics = []
for n in sorted(os.listdir("/sys/class/net")):
    if n == "lo" or n.startswith(("ifb", "veth", "docker", "br-")): continue
    spd = Path(f"/sys/class/net/{n}/speed"); st = Path(f"/sys/class/net/{n}/operstate")
    sp_ = spd.read_text().strip() if spd.exists() else "?"
    nics.append({"iface": n, "speed_Mbps": "unknown" if sp_ in ("-1", "?") else sp_, "state": st.read_text().strip() if st.exists() else "?",
                 "driver": os.path.basename(os.path.realpath(f"/sys/class/net/{n}/device/driver")) if Path(f"/sys/class/net/{n}/device/driver").exists() else "virtual"})
hw["nics"] = nics
hw["pci_net_rdma"] = _cmd("lspci 2>/dev/null | grep -iE 'ethernet|infiniband|network' ")
hw["rdma_devices"] = sorted(os.listdir("/sys/class/infiniband")) if Path("/sys/class/infiniband").is_dir() else []
hw["gpu"] = _cmd("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader 2>/dev/null || echo none")
hw["libfabric"] = _cmd("fi_info --version 2>/dev/null | head -1 || echo missing")
hw["libfabric_providers"] = sorted({l.strip(" :") for l in _cmd("fi_info -l 2>/dev/null").splitlines() if l and not l.startswith(" ")})
hw["rdma_core"] = _cmd("dpkg-query -W -f='${Version}' rdma-core 2>/dev/null || echo ?")
hw["python"] = sys.version.split()[0]
for mod in ("numpy", "torch", "vllm"):
    try: hw[mod] = __import__(mod).__version__
    except Exception: hw[mod] = "not installed"
hw["compiler"] = _cmd("gcc --version | head -1")
md = [f"# Hardware: `{HOST}` ({MODE} mode)", "", f"Captured {STAMP} by notebook 00. Measured figures in this directory come from this machine.", "",
      "| | |", "|---|---|",
      f"| Virtualization | {hw['virtualization']}" + (f" on {hw['cpu']['Hypervisor vendor']}" if hw['cpu'].get('Hypervisor vendor') else "") + " |", f"| OS / kernel | {hw['os']} / {hw['kernel']} |",
      f"| CPU | {hw['cpu'].get('Model name', '?')}; {hw['cpu'].get('CPU(s)', '?')} logical CPUs, {hw['cpu'].get('Core(s) per socket', '?')} cores x {hw['cpu'].get('Socket(s)', '?')} socket(s), {hw['cpu'].get('Thread(s) per core', '?')} thread(s)/core |",
      f"| Caches | L1d {hw['cpu'].get('L1d cache', '?')}, L2 {hw['cpu'].get('L2 cache', '?')}, L3 {hw['cpu'].get('L3 cache', '?')} |",
      f"| Memory | {hw['memory_GiB']} GiB |",
      f"| Block devices (>= 1 GiB; ROTA 1 = reported rotational) | {' / '.join(' '.join(l.split()) for l in hw['block_devices'].splitlines()) or '?'} |",
      f"| /tmp (local-mode results and spill) | {hw['tmp_mount']} |",
      "| NICs | " + (", ".join(n["iface"] + " (" + n["driver"] + ", " + str(n["speed_Mbps"]) + " Mb/s, " + n["state"] + ")" for n in nics) or "none") + " |",
      f"| RDMA devices | {', '.join(hw['rdma_devices']) or 'none'} |",
      f"| GPU | {hw['gpu']} |",
      f"| libfabric | {hw['libfabric']}; providers: {', '.join(hw['libfabric_providers'])} |",
      f"| rdma-core | {hw['rdma_core']} |",
      f"| Python / numpy / torch / vLLM | {hw['python']} / {hw['numpy']} / {hw['torch']} / {hw['vllm']} |",
      f"| Compiler | {hw['compiler']} |", ""]
if MODE == "local":
    md += ["Local mode is a smoke test: no RDMA device, libfabric `tcp` / `shm` providers in loopback, `/tmp` on a virtual disk instead of NVMe. Nothing measured here is a thesis result.", ""]
hwp = FIGDIR / MODE / "HARDWARE.md"; hwp.write_text(chr(10).join(md)); print("wrote", hwp.relative_to(ROOT)); print(chr(10).join(md))
info["hardware"] = hw

wrote docs/figures/local/HARDWARE.md
# Hardware: `vm` (local mode)

Captured 20261026T181938Z by notebook 00. Measured figures in this directory come from this machine.

| | |
|---|---|
| Virtualization | docker on KVM |
| OS / kernel | Ubuntu 24.04.5 LTS / 6.18.44-fc-v153 |
| CPU | Intel(R) Xeon(R) Processor @ 2.80GHz; 2 logical CPUs, 2 cores x 1 socket(s), 1 thread(s)/core |
| Caches | L1d 64 KiB (2 instances), L2 2 MiB (2 instances), L3 33 MiB (1 instance) |
| Memory | 7.8 GiB |
| Block devices (>= 1 GiB; ROTA 1 = reported rotational) | vda 256G disk 1 |
| /tmp (local-mode results and spill) | /dev/vda        252G   19G   25G  44% / |
| NICs | eth0 (virtio_net, unknown Mb/s, up) |
| RDMA devices | none |
| GPU | none |
| libfabric | fi_info: 1.17.0; providers: net, ofi_hook_debug, ofi_hook_dmabuf_peer_mem, ofi_hook_hmem, ofi_hook_noop, ofi_hook_perf, ofi_mrail, ofi_rxd, ofi_rxm, psm, psm2, shm, sockets, tcp, udp |
| rdma-core | 50.0-2ubuntu0.2 |
| Python / numpy / torch / vLLM | 3.1

## RDMA devices and GIDs

In [5]:
ib = Path("/sys/class/infiniband")
devs = sorted(p.name for p in ib.iterdir()) if ib.is_dir() else []
info["rdma_devices"] = devs
print("RDMA devices:", devs or "none")
if not devs:
    nd = NETDEV or "<10GbE iface>"
    print(f"No device. SoftRoCE: sudo modprobe rdma_rxe && sudo rdma link add rxe0 type rxe netdev {nd}")
gids = []
for d in devs:
    for g in sorted((ib / d / "ports/1/gids").iterdir(), key=lambda p: int(p.name)):
        val = g.read_text().strip()
        if val.strip("0:") == "":
            continue
        try:
            typ = (ib / d / "ports/1/gid_attrs/types" / g.name).read_text().strip()
        except OSError:
            typ = "?"
        gids.append({"dev": d, "index": int(g.name), "gid": val, "type": typ})
info["gids"] = gids
pd_gids = __import__("pandas").DataFrame(gids) if gids else None
pd_gids

RDMA devices: none
No device. SoftRoCE: sudo modprobe rdma_rxe && sudo rdma link add rxe0 type rxe netdev <10GbE iface>


In [6]:
for cmd in ["ibv_devices", "ibv_devinfo -v | head -60", "rdma link show"]:
    if shutil.which(cmd.split()[0]):
        info[cmd] = sh(cmd, check=False).stdout

$ ibv_devices
 Failed to get IB devices list: Function not implemented

$ ibv_devinfo -v | head -60
 Failed to get IB devices list: Function not implemented

$ rdma link show
 Failed to open NETLINK_RDMA socket



## libfabric providers and pyverbs

In [7]:
if shutil.which("fi_info"):
    info["fi_info_version"] = sh("fi_info --version", check=False).stdout
    info["fi_providers"] = [l.strip(" :") for l in sh("fi_info -l", check=False).stdout.split("\n") if l and not l.startswith(" ")]
    for p in ["verbs", "verbs;ofi_rxm", "tcp", "shm"]:
        r = sh(f"fi_info -p '{p}' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6", check=False)
        info[f"fi_info[{p}]"] = r.stdout
else:
    print("libfabric missing: sudo apt install libfabric-dev libfabric-bin")
try:
    import pyverbs.device as d
    info["pyverbs"] = [x.name.decode() for x in d.get_device_list()]
except Exception as e:
    info["pyverbs"] = f"unavailable: {e}"
print("pyverbs:", info["pyverbs"])

$ fi_info --version


fi_info: 1.17.0
libfabric: 1.17.0
libfabric api: 1.17
 
$ fi_info -l


psm2:
    version: 117.0
psm:
    version: 117.0
ofi_rxm:
    version: 117.0
ofi_rxd:
    version: 117.0
shm:
    version: 117.0
udp:
    version: 117.0
tcp:
    version: 117.0
sockets:
    version: 117.0
net:
    version: 117.0
ofi_hook_perf:
    version: 117.0
ofi_hook_debug:
    version: 117.0
ofi_hook_noop:
    version: 117.0
ofi_hook_hmem:
    version: 117.0
ofi_hook_dmabuf_peer_mem:
    version: 117.0
ofi_mrail:
    version: 117.0
 
$ fi_info -p 'verbs' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


 
$ fi_info -p 'verbs;ofi_rxm' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


 
$ fi_info -p 'tcp' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


provider: tcp;ofi_rxm
    domain: eth0
    type: FI_EP_RDM
provider: tcp;ofi_rxm
    domain: lo
    type: FI_EP_RDM
 
$ fi_info -p 'shm' -c 'FI_RMA|FI_MSG' 2>/dev/null | grep -E 'provider|type|domain' | head -6


provider: shm
    domain: shm
    type: FI_EP_RDM
provider: shm
    domain: shm
    type: FI_EP_RDM
 
pyverbs: unavailable: No module named 'pyverbs.device'


## Build the native libraries

In [8]:
for d in ["middleware/libscuffedfabric", "middleware/libscuffedverbs", "middleware/nccl_plugin"]:
    r = sh(f"make -C {d}", check=False)
    info[f"build:{d}"] = "ok" if r.returncode == 0 else r.stderr[-1500:]
    print(d, info[f"build:{d}"][:200])

$ make -C middleware/libscuffedfabric
make: Entering directory '/home/claude/scuffedrdma/middleware/libscuffedfabric'
make: Nothing to be done for 'all'.
make: Leaving directory '/home/claude/scuffedrdma/middleware/libscuffedfabric'
 
middleware/libscuffedfabric ok
$ make -C middleware/libscuffedverbs
make: Entering directory '/home/claude/scuffedrdma/middleware/libscuffedverbs'
make: Nothing to be done for 'all'.
make: Leaving directory '/home/claude/scuffedrdma/middleware/libscuffedverbs'
 
middleware/libscuffedverbs ok
$ make -C middleware/nccl_plugin
make: Entering directory '/home/claude/scuffedrdma/middleware/nccl_plugin'
make: Nothing to be done for 'all'.
make: Leaving directory '/home/claude/scuffedrdma/middleware/nccl_plugin'
 
middleware/nccl_plugin ok


## Test suite

In [9]:
if RUN_TESTS:
    r = sh("python3 -m pytest -q -p no:cacheprovider", check=False)
    info["pytest"] = r.stdout[-3000:]
    print("pytest exit", r.returncode)

$ python3 -m pytest -q -p no:cacheprovider


........................sss.sssssss..................................... [ 46%]
...................................................ss................... [ 92%]
..........ss                                                             [100%]
 
pytest exit 0


In [10]:
save("environment", info)

wrote /tmp/scuffed-local/env/vm_environment.json


PosixPath('/tmp/scuffed-local/env/vm_environment.json')